In [1]:
import sys
import xarray as xr
import numpy as np
from matplotlib import pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature
import geopandas as gpd
from shapely.geometry import mapping
from scipy.stats import spearmanr, pearsonr
import rioxarray
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

In [2]:
datap = "/Users/ellendyer/Documents/GitHub/Isotopes_F4R/plots/"
dataf = "/Users/ellendyer/Documents/GitHub/F4R_data/analysed_modis/"

### Read in MODIS evaporation  
- do this for whole available ts and then sub-select years in next step for analysis

In [3]:
Y1=2018
Y2=2018

evap_all_list = []
for Y in range(Y1,Y2+1):
    
    times = pd.read_csv('/Volumes/New_5TB/ESA_F4R/modis_evap/f4r_timestamps/f4r_dwn_'+str(Y)+'_MOD16A2GF_ET_timestamps.csv')
    times_dt = pd.to_datetime(times['timestamp'])
    
    #evap = xr.open_dataset('/Volumes/New_5TB/ESA_F4R/modis_evap/MOD16A2GF_'+str(Y)+'.tif',engine='rasterio')
    #evap = rioxarray.open_rasterio('/Volumes/New_5TB/ESA_F4R/modis_evap/MOD16A2GF_'+str(Y)+'.tif')
    
    # 1. Read the evapta into memory
    # open_rasterio reads the file into an xarray evaptaArray
    evap = rioxarray.open_rasterio('/Volumes/New_5TB/ESA_F4R/modis_evap/MOD16A2GF_'+str(Y)+'.tif')
 
    # (optional) 2. Check and enforce the native MODIS Sinusoidal CRS if it is missing.
    if evap.rio.crs is None:
        print("CRS is missing. Writing native MODIS Sinusoidal CRS...")
        modis_sinu_crs = "+proj=sinu +lon_0=0 +x_0=0 +y_0=0 +a=6371007.181 +b=6371007.181 +units=m +no_defs"
        evap = evap.rio.write_crs(modis_sinu_crs)
 
    print(f"Original CRS: {evap.rio.crs}")
 
    # 3. Reproject the data into WGS84 (EPSG:4326)
    evap = evap.rio.reproject("EPSG:4326")
    print(f"New CRS: {evap.rio.crs}")
    # The projected evapta 'da_wgs84' is now ready in memory
    # Optional: If you want to save the reprojected evapta to a new file later
    # evap_wgs84.rio.to_netcdf("MOD16A2GF_2013_WGS84.nc")
    
    evap = evap.rename(band='time')
    evap['time'] = times_dt.values
    
    print(evap)

    evap = evap.rename({'y':'lat','x':'lon'})
    evap = evap.sel(lat=slice(12,-15),lon=slice(8,31),drop=True).load()
    evap = evap.interp(lat=np.arange(-15,12,0.25), lon=np.arange(8,31,0.25), method="linear")
    evap_year_list = []
    for m in range(1,13):
        #try:
        mp = evap.sel(time=(evap.time.dt.month==m), drop=True)
        max_day = mp.time.dt.days_in_month[0].values
        bins = [str(Y)+'-'+"{:02d}".format(m)+'-01', str(Y)+'-'+"{:02d}".format(m)+'-10', str(Y)+'-'+"{:02d}".format(m)+'-20', str(Y)+'-'+"{:02d}".format(m)+'-'+str(max_day)]
        bins = pd.to_datetime(bins)
        mp_out = mp.groupby_bins('time', bins,labels=bins[1:4]).mean()
        mp_out = mp_out.rename({'time_bins':'time'})
        #print(mp_out)
        evap_year_list.append(mp_out)
        #except:
        #    print('no month - ',m,' for year - ',Y)
    evap_year = xr.concat(evap_year_list,dim='time')
    evap_year = evap_year.sortby('lat')
    evap_year = evap_year.sortby('lon')

    #evap_all_list.append(evap_year)
    del evap
    del mp_out
    print('done - ',Y)
    #evap_all = xr.concat(evap_all_list,dim='time')
    evap_year = evap_year.sel(time=slice('2018-07-01','2024-12-31'))
    evap_year_ds = evap_year.to_dataset(dim=None,name='ET_500m')
    evap_year_ds = evap_year_ds.drop('spatial_ref')
    print(evap_year_ds)
    
    evap_year_ds.to_netcdf(dataf+'modis_'+str(Y)+'_10day_reg_regrid.nc',engine='h5netcdf')
    del evap_year
    del evap_year_ds
        

Original CRS: PROJCS["MODIS Sinusoidal",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Sinusoidal"],PARAMETER["longitude_of_center",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
New CRS: EPSG:4326
<xarray.DataArray (time: 46, y: 7063, x: 11621)> Size: 15GB
array([[[ 0.00000000e+00,  2.39221739e-14,  2.09087034e-14, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 1.24334550e-16,  2.54409175e-14,  2.60196460e-14, ...,
          0.00000000e+00,  0.00000000e+00, -9.99000000e+02],
        [ 1.35397655e-15,  3.08771427e-14,  2.29143622e-14, ...,
          0.00000000e+00,  0.00000000e+00, -9.99000000e+02],
        ...,
        [ 0.00000000e+00,  0.00000000e+00,  0.0000000

In [4]:
#evap24 = xr.open_dataset(dataf+'_modis_2024_10day_reg_regrid.nc')
#print(evap24.lon)
#print(len(evap24.lon.values))

In [5]:
#evapall = xr.open_mfdataset(dataf+'modis_*_10day_reg_regrid.nc')
#print(evapall.lon)
#print(len(evapall.lon.values))

In [6]:
#evap24['lon'] = evapall['lon']
#
#merge_evap = xr.merge([evap24,evapall])*0.1/8.0
#
#print(merge_evap.lon.values)
#
#merge_evap.to_netcdf(dataf+'merged_modis_10day_reg_regrid.nc',engine='h5netcdf')